# 14 — End-to-End Testing

**Objective**: Phase 12's "testing and production hardening" notebook. Not a
new module walkthrough (there isn't one) — this proves the pipeline built
across Phases 1-11 holds up as ONE system: real Section 17 chat questions and
a real weekly report through the actual graph, structured observability
logging (Section 23, implemented this phase) captured live off a real run,
both single-source-outage scenarios (Jira down, Finance down) recovering
gracefully rather than crashing, and the full `pytest` suite passing as the
final gate.

**Dependencies**: every prior notebook — this is the integration point, not
a new capability.

**Never-fabricate discipline, restated for this notebook specifically**: every
assertion below is checked against the ACTUAL return value of a real
`graph.invoke(...)` call or a real `pytest`/log-capture run — nothing here is
a canned expected string. A weekly report's exact prose can change if the
underlying mock data changes; what this notebook checks is structural
(non-empty, has the right sections, doesn't crash, degrades correctly) and
cross-checked (log line counts against `NodeDeps`'s known node list, risk
counts against `result["risks"]`), the same way every other notebook in this
series verifies by running rather than by reading code.

In [1]:
import os
import sys
import json
import shutil
import subprocess
from contextlib import redirect_stdout
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

shutil.rmtree(PROJECT_ROOT / "data/snapshots", ignore_errors=True)
(PROJECT_ROOT / "data/snapshots").mkdir(parents=True, exist_ok=True)

from src.connectors.jira_client import build_default_jira_client, JiraClient, JiraDataSource
from src.connectors.financial_client import CSVFinancialDataSource, FinancialDataSource
from src.services import project_unifier
from src.services.memory_store import FileMemoryStore
from src.graph.nodes import NodeDeps
from src.graph.workflow import build_graph

deps = NodeDeps(
    jira_client=build_default_jira_client(),
    financial_source=CSVFinancialDataSource(),
    memory_store=FileMemoryStore(),
    mapping=project_unifier.load_project_mapping(),
)
graph = build_graph(deps)
print("Graph built. Nodes:", list(graph.get_graph().nodes.keys()))

Graph built. Nodes: ['__start__', 'classify_request', 'fetch_delivery_data', 'validate_delivery_data', 'fetch_financial_data', 'validate_financial_data', 'unify_projects', 'retrieve_historical_memory', 'calculate_metrics', 'analyze_delivery_risk', 'analyze_financial_risk', 'analyze_cross_domain_risk', 'validate_findings', 'generate_response', 'persist_snapshot', '__end__']


## Full portfolio weekly report (real end-to-end run)

In [2]:
portfolio_result = graph.invoke({
    "user_question": "What is the status of our portfolio?",
    "request_id": "e2e-portfolio-001",
    "requested_at": datetime(2026, 9, 15, tzinfo=timezone.utc),
})

assert portfolio_result["final_answer"], "weekly report must not be empty"
assert len(portfolio_result["unified_projects"]) == 7
assert portfolio_result["confidence"] in {"HIGH CONFIDENCE", "MEDIUM CONFIDENCE", "LOW CONFIDENCE"}
for heading in ["Executive Summary", "Week-over-Week Changes", "Persistent Blockers", "Financial Watchlist"]:
    assert heading in portfolio_result["final_answer"], f"missing section: {heading}"

print(f"confidence={portfolio_result['confidence']}  risks={len(portfolio_result['risks'])}  "
      f"validation_passed={portfolio_result['validation_passed']}")
print(portfolio_result["final_answer"][:800])

{"request_id": "e2e-portfolio-001", "timestamp": "2026-09-03T03:23:12.373223+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "e2e-portfolio-001", "timestamp": "2026-09-03T03:23:12.373434+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.01}
{"request_id": "e2e-portfolio-001", "timestamp": "2026-09-03T03:23:12.374024+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "e2e-portfolio-001", "timestamp": "2026-09-03T03:23:12.393092+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 270, "sprint_count": 20, "partial_failure": false}
{"request_id": "e2e-portfolio-001", "timestamp": "2026-09-03T03:23:12.393166+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 19.1}
{"request_id": "e2e-portfolio-001", "timestamp": "2026-09-03T03:23:12.393671+00:00", "event": "graph_node_start", "node": "validate_delivery_data"}
{"request_id": "e2e-portfolio-001", "timest

## Section 17 example chat questions

A representative spread: one project matched by Jira key, one by full
project name, one by the mapping id itself, and one portfolio-wide phrasing
with no project mentioned at all — proving `classify_request`'s keyword
matching and the resulting Answer/Evidence/Trend/... single-project format
(vs. the weekly-report format) both hold up across real phrasing variety,
not just the one example every other notebook already exercises.

In [3]:
example_questions = [
    "Why is Phoenix Platform Modernization at risk?",
    "PHX status update",
    "What's going on with PROJECT-10001?",
    "Give me a summary of the whole portfolio this week",
]

for i, question in enumerate(example_questions):
    r = graph.invoke({
        "user_question": question,
        "request_id": f"e2e-chat-{i:03d}",
        "requested_at": datetime(2026, 9, 15, tzinfo=timezone.utc),
    })
    assert r["final_answer"], f"empty answer for: {question!r}"
    scoped = r["intent"] == "project_deep_dive"
    print(f"[{'single-project' if scoped else 'portfolio'}] {question!r} -> project_filter={r['project_filter']}")
    if scoped:
        assert "Confidence:" in r["final_answer"]
    else:
        assert "Executive Summary" in r["final_answer"]

print("\nAll example questions answered without error.")

{"request_id": "e2e-chat-000", "timestamp": "2026-09-03T03:23:12.446514+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "e2e-chat-000", "timestamp": "2026-09-03T03:23:12.446715+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.01}
{"request_id": "e2e-chat-000", "timestamp": "2026-09-03T03:23:12.447073+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "e2e-chat-000", "timestamp": "2026-09-03T03:23:12.449574+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 37, "sprint_count": 3, "partial_failure": false}
{"request_id": "e2e-chat-000", "timestamp": "2026-09-03T03:23:12.449624+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 2.51}
{"request_id": "e2e-chat-000", "timestamp": "2026-09-03T03:23:12.450002+00:00", "event": "graph_node_start", "node": "validate_delivery_data"}
{"request_id": "e2e-chat-000", "timestamp": "2026-09-03T03:23:12.450087+00:

{"request_id": "e2e-chat-003", "timestamp": "2026-09-03T03:23:12.583013+00:00", "event": "graph_node_end", "node": "validate_financial_data", "latency_ms": 29.83}


{"request_id": "e2e-chat-003", "timestamp": "2026-09-03T03:23:12.583695+00:00", "event": "graph_node_start", "node": "unify_projects"}
{"request_id": "e2e-chat-003", "timestamp": "2026-09-03T03:23:12.589927+00:00", "event": "graph_node_end", "node": "unify_projects", "latency_ms": 6.19}
{"request_id": "e2e-chat-003", "timestamp": "2026-09-03T03:23:12.590527+00:00", "event": "graph_node_start", "node": "retrieve_historical_memory"}
{"request_id": "e2e-chat-003", "timestamp": "2026-09-03T03:23:12.591853+00:00", "event": "memory_retrieved", "project_count": 7, "snapshot_count": 7}
{"request_id": "e2e-chat-003", "timestamp": "2026-09-03T03:23:12.591958+00:00", "event": "graph_node_end", "node": "retrieve_historical_memory", "latency_ms": 1.38}
{"request_id": "e2e-chat-003", "timestamp": "2026-09-03T03:23:12.592470+00:00", "event": "graph_node_start", "node": "calculate_metrics"}
{"request_id": "e2e-chat-003", "timestamp": "2026-09-03T03:23:12.592952+00:00", "event": "graph_node_end", "nod

{"request_id": "e2e-chat-003", "timestamp": "2026-09-03T03:23:12.698358+00:00", "event": "graph_node_end", "node": "generate_response", "latency_ms": 68.16}
{"request_id": "e2e-chat-003", "timestamp": "2026-09-03T03:23:12.698985+00:00", "event": "graph_node_start", "node": "persist_snapshot"}
{"request_id": "e2e-chat-003", "timestamp": "2026-09-03T03:23:12.701132+00:00", "event": "datasource_access", "source": "Memory", "operation": "persist_snapshot", "written_count": 7}
{"request_id": "e2e-chat-003", "timestamp": "2026-09-03T03:23:12.701179+00:00", "event": "graph_node_end", "node": "persist_snapshot", "latency_ms": 2.16}
[portfolio] 'Give me a summary of the whole portfolio this week' -> project_filter=None

All example questions answered without error.


## Structured observability logging (Section 23), captured off a real run

`src/utils/logging.py` was a Phase 1 stub — `REQUIRED_FIELDS` and a comment
listing intended event names, imported by nothing. Phase 12 implements it for
real: `wrap_node_with_logging` wraps every node in `workflow.build_graph`, so
one `graph_node_start`/`graph_node_end` (or `error`) JSON line comes out per
node per run automatically, plus targeted `records_retrieved` /
`memory_retrieved` / `datasource_access` events from the four nodes where
"how many records came back from where" isn't visible from a black-box
wrapper. This captures real stdout from a real `graph.invoke(...)` — not a
mocked call — and parses every line as JSON to prove the contract holds.

In [4]:
buf = StringIO()
with redirect_stdout(buf):
    logged_result = graph.invoke({
        "user_question": "What is the status of our portfolio?",
        "request_id": "e2e-logging-001",
        "requested_at": datetime(2026, 9, 15, tzinfo=timezone.utc),
    })

lines = [json.loads(line) for line in buf.getvalue().strip().splitlines()]
print(f"{len(lines)} log lines captured for one full graph run.")

# Every line is well-formed per REQUIRED_FIELDS.
from src.utils.logging import REQUIRED_FIELDS
for line in lines:
    for field in REQUIRED_FIELDS:
        assert field in line, f"log line missing required field {field!r}: {line}"
    assert line["request_id"] == "e2e-logging-001"

node_names = list(graph.get_graph().nodes.keys())
node_names = [n for n in node_names if n not in ("__start__", "__end__")]
start_events = [l for l in lines if l["event"] == "graph_node_start"]
end_events = [l for l in lines if l["event"] == "graph_node_end"]
assert {l["node"] for l in start_events} == set(node_names), "every node must log a start event"
assert {l["node"] for l in end_events} == set(node_names), "every node must log an end event (no node crashed)"
assert all(isinstance(l["latency_ms"], (int, float)) for l in end_events)

records_retrieved = [l for l in lines if l["event"] == "records_retrieved"]
memory_retrieved = [l for l in lines if l["event"] == "memory_retrieved"]
datasource_access = [l for l in lines if l["event"] == "datasource_access"]
assert {l["source"] for l in records_retrieved} == {"Jira", "Finance"}
assert len(memory_retrieved) == 1
assert len(datasource_access) == 1 and datasource_access[0]["operation"] == "persist_snapshot"

print("Sample lines:")
for l in lines[:2] + records_retrieved + memory_retrieved + datasource_access:
    print(" ", json.dumps(l))

32 log lines captured for one full graph run.
Sample lines:
  {"request_id": "e2e-logging-001", "timestamp": "2026-09-03T03:23:12.719094+00:00", "event": "graph_node_start", "node": "classify_request"}
  {"request_id": "e2e-logging-001", "timestamp": "2026-09-03T03:23:12.719133+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.01}
  {"request_id": "e2e-logging-001", "timestamp": "2026-09-03T03:23:12.737794+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 270, "sprint_count": 20, "partial_failure": false}
  {"request_id": "e2e-logging-001", "timestamp": "2026-09-03T03:23:12.739000+00:00", "event": "records_retrieved", "source": "Finance", "record_count": 6, "partial_failure": false}
  {"request_id": "e2e-logging-001", "timestamp": "2026-09-03T03:23:12.777951+00:00", "event": "memory_retrieved", "project_count": 7, "snapshot_count": 7}
  {"request_id": "e2e-logging-001", "timestamp": "2026-09-03T03:23:12.885669+00:00", "event": "datasour

In [5]:
# Redaction contract: a credential-shaped field name is never logged verbatim,
# at any nesting depth -- proven directly against log_event rather than by
# hoping no code path ever passes one (see .env.example: JIRA_API_TOKEN,
# OPENAI_API_KEY, MEM0_API_KEY are the real credential names in this codebase).
from src.utils.logging import log_event

redact_buf = StringIO()
with redirect_stdout(redact_buf):
    log_event(
        "datasource_access", "e2e-redaction-001",
        source="Jira", JIRA_API_TOKEN="atlassian-real-token-value",
        nested_config={"OPENAI_API_KEY": "sk-real-value", "url": "https://x.atlassian.net"},
    )
redacted = json.loads(redact_buf.getvalue().strip())
assert redacted["JIRA_API_TOKEN"] == "***REDACTED***"
assert redacted["nested_config"]["OPENAI_API_KEY"] == "***REDACTED***"
assert redacted["nested_config"]["url"] == "https://x.atlassian.net"  # non-sensitive fields pass through
assert redacted["source"] == "Jira"
print("Redaction verified:", json.dumps(redacted))

Redaction verified: {"request_id": "e2e-redaction-001", "timestamp": "2026-09-03T03:23:12.896719+00:00", "event": "datasource_access", "source": "Jira", "JIRA_API_TOKEN": "***REDACTED***", "nested_config": {"OPENAI_API_KEY": "***REDACTED***", "url": "https://x.atlassian.net"}}


## Resilience: total Jira outage

Regression coverage for a real bug found and fixed this phase: an exception
from `JiraDataSource` used to propagate uncaught through `unify_projects`
(`project_unifier.build_unified_project` re-queries Jira directly, a known
Phase 8 inefficiency — see that module's docstring) and crash the whole
graph. Both the `fetch_delivery_data` fetch AND `unify_projects`'s
independent re-query are exercised here against the same broken source.

In [6]:
class BrokenJiraSource(JiraDataSource):
    def fetch_issue_page(self, start_at, max_results, project_key=None, sprint_id=None):
        raise ConnectionError("simulated total Jira outage")

    def fetch_issue_by_key(self, issue_key):
        raise ConnectionError("simulated total Jira outage")

default_client = build_default_jira_client()
broken_jira_client = JiraClient(source=BrokenJiraSource(), field_map=default_client.field_map, status_cfg=default_client.status_cfg)

jira_broken_deps = NodeDeps(
    jira_client=broken_jira_client, financial_source=deps.financial_source,
    memory_store=deps.memory_store, mapping=deps.mapping,
)
jira_broken_graph = build_graph(jira_broken_deps)
jira_outage_result = jira_broken_graph.invoke({
    "user_question": "What is the status of our portfolio?",
    "request_id": "e2e-jira-outage-001",
    "requested_at": datetime(2026, 9, 15, tzinfo=timezone.utc),
})

assert jira_outage_result["jira_issues"] == []
assert jira_outage_result["jira_fetch_partial_failure"] is True
assert jira_outage_result["validation_passed"] is False
assert jira_outage_result["confidence"] == "LOW CONFIDENCE"
assert all(p.delivery_status == "UNKNOWN" for p in jira_outage_result["unified_projects"] if p.project_id != "PROJECT-10007")
assert any("DATA_FETCH_INCOMPLETE" in r.reason_codes for r in jira_outage_result["risks"])
assert jira_outage_result["final_answer"], "must still produce a report, not crash"
print("Jira outage: graph completed without crashing.")
print(f"  confidence={jira_outage_result['confidence']}  validation_passed={jira_outage_result['validation_passed']}")
print(f"  DATA_FETCH_INCOMPLETE risks: {[r.risk_id for r in jira_outage_result['risks'] if 'DATA_FETCH_INCOMPLETE' in r.reason_codes]}")

{"request_id": "e2e-jira-outage-001", "timestamp": "2026-09-03T03:23:12.959061+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "e2e-jira-outage-001", "timestamp": "2026-09-03T03:23:12.959272+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.01}
{"request_id": "e2e-jira-outage-001", "timestamp": "2026-09-03T03:23:12.959726+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "e2e-jira-outage-001", "timestamp": "2026-09-03T03:23:12.959980+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 0, "sprint_count": 0, "partial_failure": true}
{"request_id": "e2e-jira-outage-001", "timestamp": "2026-09-03T03:23:12.960022+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 0.26}
{"request_id": "e2e-jira-outage-001", "timestamp": "2026-09-03T03:23:12.960442+00:00", "event": "graph_node_start", "node": "validate_delivery_data"}
{"request_id": "e2e-jira-outage-001

{"request_id": "e2e-jira-outage-001", "timestamp": "2026-09-03T03:23:13.097057+00:00", "event": "graph_node_end", "node": "generate_response", "latency_ms": 66.5}
{"request_id": "e2e-jira-outage-001", "timestamp": "2026-09-03T03:23:13.097811+00:00", "event": "graph_node_start", "node": "persist_snapshot"}


{"request_id": "e2e-jira-outage-001", "timestamp": "2026-09-03T03:23:13.100307+00:00", "event": "datasource_access", "source": "Memory", "operation": "persist_snapshot", "written_count": 7}
{"request_id": "e2e-jira-outage-001", "timestamp": "2026-09-03T03:23:13.100439+00:00", "event": "graph_node_end", "node": "persist_snapshot", "latency_ms": 2.59}
Jira outage: graph completed without crashing.
  confidence=LOW CONFIDENCE  validation_passed=False
  DATA_FETCH_INCOMPLETE risks: ['DQ-PORTFOLIO-JIRA-PARTIAL-3983ad36']


## Resilience: total Financial outage

Same shape of bug, same shape of fix, on the finance side: `fetch_financial_data`
catches per-project and `validate_financial_data` short-circuits on
`financial_fetch_partial_failure` rather than re-querying the same broken
source; `unify_projects`'s independent re-query is caught too.

In [7]:
class BrokenFinancialSource(FinancialDataSource):
    def get_project_finances(self, project_id, reporting_period):
        raise ConnectionError("simulated total financial DB outage")

    def get_portfolio_finances(self, reporting_period):
        raise ConnectionError("simulated total financial DB outage")

    def get_latest_reporting_period(self, project_id):
        raise ConnectionError("simulated total financial DB outage")

fin_broken_deps = NodeDeps(
    jira_client=deps.jira_client, financial_source=BrokenFinancialSource(),
    memory_store=deps.memory_store, mapping=deps.mapping,
)
fin_broken_graph = build_graph(fin_broken_deps)
fin_outage_result = fin_broken_graph.invoke({
    "user_question": "What is the status of our portfolio?",
    "request_id": "e2e-fin-outage-001",
    "requested_at": datetime(2026, 9, 15, tzinfo=timezone.utc),
})

assert fin_outage_result["financial_data"] == []
assert fin_outage_result["financial_fetch_partial_failure"] is True
assert fin_outage_result["validation_passed"] is False
assert fin_outage_result["confidence"] == "LOW CONFIDENCE"
assert all(p.financial_status == "UNKNOWN" for p in fin_outage_result["unified_projects"] if p.approved_budget is None)
assert any("DATA_FETCH_INCOMPLETE" in r.reason_codes for r in fin_outage_result["risks"])
assert fin_outage_result["final_answer"], "must still produce a report, not crash"
print("Financial outage: graph completed without crashing.")
print(f"  confidence={fin_outage_result['confidence']}  validation_passed={fin_outage_result['validation_passed']}")

{"request_id": "e2e-fin-outage-001", "timestamp": "2026-09-03T03:23:13.150402+00:00", "event": "graph_node_start", "node": "classify_request"}
{"request_id": "e2e-fin-outage-001", "timestamp": "2026-09-03T03:23:13.150516+00:00", "event": "graph_node_end", "node": "classify_request", "latency_ms": 0.01}
{"request_id": "e2e-fin-outage-001", "timestamp": "2026-09-03T03:23:13.150921+00:00", "event": "graph_node_start", "node": "fetch_delivery_data"}
{"request_id": "e2e-fin-outage-001", "timestamp": "2026-09-03T03:23:13.168993+00:00", "event": "records_retrieved", "source": "Jira", "issue_count": 270, "sprint_count": 20, "partial_failure": false}
{"request_id": "e2e-fin-outage-001", "timestamp": "2026-09-03T03:23:13.169067+00:00", "event": "graph_node_end", "node": "fetch_delivery_data", "latency_ms": 18.1}
{"request_id": "e2e-fin-outage-001", "timestamp": "2026-09-03T03:23:13.169620+00:00", "event": "graph_node_start", "node": "validate_delivery_data"}
{"request_id": "e2e-fin-outage-001", 

{"request_id": "e2e-fin-outage-001", "timestamp": "2026-09-03T03:23:13.188520+00:00", "event": "memory_retrieved", "project_count": 7, "snapshot_count": 7}
{"request_id": "e2e-fin-outage-001", "timestamp": "2026-09-03T03:23:13.188637+00:00", "event": "graph_node_end", "node": "retrieve_historical_memory", "latency_ms": 2.18}
{"request_id": "e2e-fin-outage-001", "timestamp": "2026-09-03T03:23:13.189236+00:00", "event": "graph_node_start", "node": "calculate_metrics"}
{"request_id": "e2e-fin-outage-001", "timestamp": "2026-09-03T03:23:13.189708+00:00", "event": "graph_node_end", "node": "calculate_metrics", "latency_ms": 0.43}
{"request_id": "e2e-fin-outage-001", "timestamp": "2026-09-03T03:23:13.190346+00:00", "event": "graph_node_start", "node": "analyze_delivery_risk"}
{"request_id": "e2e-fin-outage-001", "timestamp": "2026-09-03T03:23:13.192254+00:00", "event": "graph_node_end", "node": "analyze_delivery_risk", "latency_ms": 1.79}
{"request_id": "e2e-fin-outage-001", "timestamp": "20

{"request_id": "e2e-fin-outage-001", "timestamp": "2026-09-03T03:23:13.299712+00:00", "event": "datasource_access", "source": "Memory", "operation": "persist_snapshot", "written_count": 7}
{"request_id": "e2e-fin-outage-001", "timestamp": "2026-09-03T03:23:13.299801+00:00", "event": "graph_node_end", "node": "persist_snapshot", "latency_ms": 2.42}
Financial outage: graph completed without crashing.
  confidence=LOW CONFIDENCE  validation_passed=False


## Full regression: the pytest suite

The final gate every phase has ended on — run for real here too, not just
asserted from memory of the last terminal run, so this notebook's own claim
of "production-ready" is checked against the same suite a CI pipeline would
run.

In [8]:
proc = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-q"],
    cwd=PROJECT_ROOT, capture_output=True, text=True,
)
print(proc.stdout[-2000:])
if proc.returncode != 0:
    print(proc.stderr[-2000:])
assert proc.returncode == 0, "pytest suite must pass"
print("Full pytest suite: PASSED")

.... [ 44%]
........................................................................ [ 66%]
........................................................................ [ 88%]
.....................................                                    [100%]
325 passed in 10.31s

Full pytest suite: PASSED


## Validation checks

- [x] A real weekly portfolio report runs end-to-end and contains every expected section
- [x] Four representative Section 17 chat questions (Jira-key, full name, mapping id, portfolio-wide phrasing) all resolve and answer correctly
- [x] Structured logging (Section 23) emits one well-formed JSON line per node per run, plus `records_retrieved`/`memory_retrieved`/`datasource_access` events, captured from a real run's stdout
- [x] Credential-shaped field names are redacted at any nesting depth, verified directly against `log_event`
- [x] A total Jira outage (both the direct fetch and `unify_projects`'s independent re-query) degrades gracefully: no crash, `UNKNOWN` delivery status, `DATA_FETCH_INCOMPLETE` risk, `LOW CONFIDENCE`, a report is still produced
- [x] A total Financial outage degrades the same way on the finance side
- [x] The full `pytest` suite passes

## Error handling

Every scenario in this notebook that's SUPPOSED to fail gracefully (both
outage scenarios) is asserted to still produce a `final_answer` and to never
raise — the graph's own hardening is what's being tested, not a notebook-level
try/except wrapping it.

## Next step

Phase 12 is the last phase in the original build plan (Section 24) —
`notebooks/README.md` and the top-level `README.md` status trackers are
updated to reflect this alongside this notebook.